In [ ]:
# Define categorize_income to avoid NameError
def categorize_income(income):
    if income < 30000:
        return 'low'
    elif income < 60000:
        return 'average'
    elif income < 100000:
        return 'high'
    else:
        return 'extreme'

# ...existing code...

import numpy as np
import pandas as pd

def safe_rmsle(y_true, y_pred):
    """Calculate the Root Mean Squared Logarithmic Error (RMSLE) in a safe manner."""
    # Ensure predictions have the correct shape
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Debug: Print statistics of the true and predicted values
    print("True values statistics:")
    print(pd.Series(y_true).describe())
    print("Predicted values statistics:")
    print(pd.Series(y_pred).describe())
    
    # Check for invalid values
    if np.any(np.isnan(y_true)) or np.any(np.isnan(y_pred)):
        print("NaNs found in true or predicted values")
        return np.inf
    if np.any(np.isinf(y_true)) or np.any(np.isinf(y_pred)):
        print("Infs found in true or predicted values")
        return np.inf
    
    # Calculate RMSLE
    log_true = np.log1p(y_true)
    log_pred = np.log1p(y_pred)
    rmsle = np.sqrt(np.mean((log_true - log_pred) ** 2))
    
    return rmsle

In [ ]:
import os
import sys
import joblib
from matplotlib import pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.metrics import mean_squared_log_error

# Load the best model
project_root = "/mnt/c/home/USER/ensemble"
model_path = os.path.join(project_root, "output", "Svr", "model", "best_model.joblib")
model = joblib.load(model_path)

# Load the validation data
data_path = os.path.join(project_root, "data", "processed", "final_validate.csv")
data = pd.read_csv(data_path)

# Categorize income and count samples in each group
data['income_group'] = data['AnnualIncome'].apply(categorize_income)
income_group_counts = data['income_group'].value_counts().sort_index()
print("Income Group Counts:")
print(income_group_counts)

target_column = "PremiumAmount"
X = data.drop([target_column, 'income_group'], axis=1)
y_true = data[target_column]

# Make predictions and compute absolute errors
y_pred = model.predict(X)
errors = np.abs(y_pred - y_true)

# Calculate RMSLE
rmsle_score = safe_rmsle(y_true, y_pred)
print(f"RMSLE Score: {rmsle_score}")

# Determine the threshold for poorly estimated samples (top 10% error)
error_threshold = np.percentile(errors, 90)
poor_samples_mask = errors >= error_threshold
poor_samples = data[poor_samples_mask].copy()

# Select a sample with the maximum error for explanation
idx = errors.argmax()
sample = X.iloc[[idx]]  # must be 2D for both explainers

# ---- SHAP Explanation ----
background = X.sample(n=100, random_state=42)
explainer = shap.KernelExplainer(model.predict, background)
shap_values = explainer.shap_values(sample, nsamples=100)

shap.initjs()
plt.figure(figsize=(12, 8))
shap.force_plot(explainer.expected_value, shap_values, sample, matplotlib=True)
plt.title("SHAP Force Plot")
plt.show()

# Additional SHAP plots
plt.figure(figsize=(12, 30))
shap.summary_plot(shap_values, sample, max_display=20)

# If shap_values is a list, extract the first element
if isinstance(shap_values, list):
    shap_vals = shap_values[0].flatten()
else:
    shap_vals = shap_values.flatten()

plt.figure(figsize=(12, 30))
shap.bar_plot(shap_vals, feature_names=sample.columns, max_display=20)

# ---- LIME Explanation ----
explainer_lime = LimeTabularExplainer(
    training_data=np.array(X),
    feature_names=X.columns,
    mode='regression',
    verbose=True,
    random_state=42
)

lime_exp = explainer_lime.explain_instance(
    data_row=sample.iloc[0],
    predict_fn=model.predict,
    num_features=len(X.columns)
)

print(lime_exp.as_list())
lime_exp.save_to_file("lime_explanation.html")
print("LIME explanation saved to lime_explanation.html")

In [ ]:
target_column = "PremiumAmount"
X = data.drop(target_column, axis=1)
y_true = data[target_column]

# Make predictions and compute absolute errors
y_pred = model.predict(X)
errors = np.abs(y_pred - y_true)

# Determine the threshold for poorly estimated samples (top 10% error)
error_threshold = np.percentile(errors, 90)
poor_samples_mask = errors >= error_threshold
poor_samples = data[poor_samples_mask].copy()

# Select a sample with the maximum error for explanation
idx = errors.argmax()
sample = X.iloc[[idx]]  # must be 2D for both explainers

# ---- SHAP Explanation ----
background = X.sample(n=100, random_state=42)
explainer = shap.KernelExplainer(model.predict, background)
shap_values = explainer.shap_values(sample, nsamples=100)

shap.initjs()
plt.figure(figsize=(12, 8))
shap.force_plot(explainer.expected_value, shap_values, sample, matplotlib=True)
plt.title("SHAP Force Plot")
plt.show()

# Additional SHAP plots
plt.figure(figsize=(12, 30))
shap.summary_plot(shap_values, sample, max_display=20)

# If shap_values is a list, extract the first element
if isinstance(shap_values, list):
    shap_vals = shap_values[0].flatten()
else:
    shap_vals = shap_values.flatten()

plt.figure(figsize=(12, 30))
shap.bar_plot(shap_vals, feature_names=sample.columns, max_display=20)

# ---- LIME Explanation ----
explainer_lime = LimeTabularExplainer(
    training_data=np.array(X),
    feature_names=X.columns,
    mode='regression',
    verbose=True,
    random_state=42
)

lime_exp = explainer_lime.explain_instance(
    data_row=sample.iloc[0],
    predict_fn=model.predict,
    num_features=len(X.columns)
)

print(lime_exp.as_list())
lime_exp.save_to_file("lime_explanation.html")
print("LIME explanation saved to lime_explanation.html")

In [ ]:
%matplotlib inline